In [1]:
import pandas as pd

In [2]:
df_raw = pd.read_csv('./data/cirrhosis.csv').drop(columns=["ID", "N_Days"])

target_col = 'Status'

X = df_raw.drop(target_col, axis=1)
y = df_raw[target_col]

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.naive_bayes import GaussianNB
from utils import create_evaluation_dataframe, Metric

In [4]:
from sklearn.preprocessing import LabelEncoder

cat_cols = X.select_dtypes(exclude=["number"]).columns
df_raw = df_raw.dropna(subset=cat_cols)

X = df_raw.drop(target_col, axis=1)
y = df_raw[target_col]
y = LabelEncoder().fit_transform(y)

print(f"Kształt danych po usunięciu braków: {X.shape}")

Kształt danych po usunięciu braków: (312, 17)


In [5]:
X_train_cl_tmp, X_test, y_train_cl_tmp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_cl_tmp, y_train_cl_tmp, test_size=0.15/0.85, random_state=42)

print(f"X_train: {X_train.shape}")
print(f"X_val: {X_val.shape}")
print(f"X_test: {X_test.shape}")

X_train: (218, 17)
X_val: (47, 17)
X_test: (47, 17)


In [6]:
num_pipelines = {
    "knn_imputation": Pipeline([
        ("imputer", KNNImputer())
    ])
}

cat_pipelines = {
    "one_hot": Pipeline([
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])
}

In [7]:
from sklearn.tree import DecisionTreeClassifier
import itertools

criteria = ['gini', 'entropy']
max_depths = [None, 3, 5, 10, 15]
max_features = [None, 'sqrt', 'log2']
ccp_alphas = [0.0, 0.005, 0.01, 0.02]

models = {}

for crit, depth, feat, alpha in itertools.product(criteria, max_depths, max_features, ccp_alphas):
    model_name = f"paras: {crit}_d{depth}_mf{feat}_a{alpha}"
    models[model_name] = DecisionTreeClassifier(
        criterion=crit,
        max_depth=depth,
        max_features=feat,
        ccp_alpha=alpha,
        random_state=42
    )

# Ewaluacja modeli
results_df = create_evaluation_dataframe(
    X_train,
    y_train,
    X_val,
    y_val,
    num_pipelines,
    cat_pipelines,
    models,
    Metric.F1_SCORE
)

display(results_df)

,num_pipeline,cat_pipeline,model,train_accuracy,val_accuracy,train_precision,val_precision,train_recall,val_recall,train_f1,val_f1
0,knn_imputation,one_hot,paras: entropy_d3_mfNone_a0.0,0.7798,0.6809,0.7369,0.6371,0.7798,0.6809,0.7577,0.6582
1,knn_imputation,one_hot,paras: entropy_d3_mfNone_a0.005,0.7798,0.6809,0.7369,0.6371,0.7798,0.6809,0.7577,0.6582
2,knn_imputation,one_hot,paras: entropy_d3_mfNone_a0.01,0.7798,0.6809,0.7369,0.6371,0.7798,0.6809,0.7577,0.6582
3,knn_imputation,one_hot,paras: entropy_d3_mfNone_a0.02,0.7798,0.6809,0.7369,0.6371,0.7798,0.6809,0.7577,0.6582
4,knn_imputation,one_hot,paras: gini_dNone_mfsqrt_a0.01,0.8440,0.6596,0.8433,0.6378,0.8440,0.6596,0.8396,0.6369
...,...,...,...,...,...,...,...,...,...,...,...
115,knn_imputation,one_hot,paras: entropy_dNone_mflog2_a0.02,0.8028,0.5106,0.8272,0.5400,0.8028,0.5106,0.8011,0.5137
116,knn_imputation,one_hot,paras: entropy_d15_mflog2_a0.02,0.8028,0.5106,0.8272,0.5400,0.8028,0.5106,0.8011,0.5137
117,knn_imputation,one_hot,paras: entropy_d15_mfsqrt_a0.02,0.8028,0.5106,0.8272,0.5400,0.8028,0.5106,0.8011,0.5137
118,knn_imputation,one_hot,paras: entropy_d10_mfsqrt_a0.01,0.9587,0.5106,0.9591,0.5242,0.9587,0.5106,0.9588,0.5128


In [8]:
best_model = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=3,
    max_features=None,
    ccp_alpha=0.0,
    random_state=42
)

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipelines["knn_imputation"], X_train.select_dtypes(include=["number"]).columns),
        ("cat", cat_pipelines["one_hot"], X_train.select_dtypes(exclude=["number"]).columns),
    ]
)

final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", best_model)
])

final_pipeline.fit(X_train, y_train)

y_train_pred = final_pipeline.predict(X_train)
y_val_pred = final_pipeline.predict(X_val)
y_test_pred = final_pipeline.predict(X_test)

def print_metrics(y_true, y_pred, set_name):
    print(f"--- {set_name} ---")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"F1 Score:  {f1_score(y_true, y_pred, average='weighted', zero_division=0):.4f}\n")

print_metrics(y_train, y_train_pred, "Train")
print_metrics(y_val, y_val_pred, "Validation")
print_metrics(y_test, y_test_pred, "Test")

--- Train ---
Accuracy:  0.7798
Precision: 0.7369
Recall:    0.7798
F1 Score:  0.7577

--- Validation ---
Accuracy:  0.6809
Precision: 0.6371
Recall:    0.6809
F1 Score:  0.6582

--- Test ---
Accuracy:  0.7234
Precision: 0.6777
Recall:    0.7234
F1 Score:  0.6913

